# KDD 2026 · dLLM Hands-on Tutorial

We begin with one default inference run. Only after it works do we inspect defaults, change one parameter at a time, and study the denoising trajectory.

**Prerequisite:** clone [dLLM](https://github.com/ZHZisZZ/dllm), check out the tutorial release, and run `pip install -e .` in that repository.

## 01 · First inference with defaults

**Goal:** load the official Tiny-A2D model and generate one answer using dLLM’s default sampler configuration.

In [ ]:
from dataclasses import asdict
import time

import transformers
import dllm

MODEL_ID = "dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1"
PROMPT = "Explain why parallel denoising is useful in one sentence."
SEED = 2026

transformers.set_seed(SEED)
model_args = dllm.utils.ModelArguments(model_name_or_path=MODEL_ID)
model = dllm.utils.get_model(model_args=model_args).eval()
tokenizer = dllm.utils.get_tokenizer(model_args=model_args)

sampler = dllm.core.samplers.MDLMSampler(model=model, tokenizer=tokenizer)
sampler_config = dllm.core.samplers.MDLMSamplerConfig()  # use every official default

messages = [[{"role": "user", "content": PROMPT}]]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True
)

started_at = time.perf_counter()
default_output = sampler.sample(inputs, sampler_config, return_dict=True)
default_latency = time.perf_counter() - started_at
default_text = dllm.utils.sample_trim(
    tokenizer, default_output.sequences.tolist(), inputs
)[0].strip()

print(default_text)
print(f"Latency: {default_latency:.2f}s")

In [ ]:
# Inspect defaults only after the first successful run.
asdict(sampler_config)

## 02 · Change one sampler choice at a time

Keep the model, prompt, and seed fixed. Each new `MDLMSamplerConfig` below specifies only the value we want to change; every omitted field keeps its official default.

In [ ]:
experiment_configs = {
    "default": dllm.core.samplers.MDLMSamplerConfig(),
    "more_steps": dllm.core.samplers.MDLMSamplerConfig(steps=256),
    "random_remasking": dllm.core.samplers.MDLMSamplerConfig(remasking="random"),
}

def run_inference(name, config):
    transformers.set_seed(SEED)
    started_at = time.perf_counter()
    output = sampler.sample(inputs, config, return_dict=True)
    text = dllm.utils.sample_trim(
        tokenizer, output.sequences.tolist(), inputs
    )[0].strip()
    return {
        "name": name,
        "config": asdict(config),
        "latency_s": round(time.perf_counter() - started_at, 3),
        "text": text,
    }

comparison_results = [
    run_inference(name, config)
    for name, config in experiment_configs.items()
]

for result in comparison_results:
    print(f"\n[{result['name']}] {result['latency_s']}s")
    print(result["text"])

In [ ]:
# Read the real denoising history returned by the default run.
def first_sequence(snapshot):
    values = snapshot.detach().cpu().tolist()
    while values and isinstance(values[0], list):
        values = values[0]
    return values

prompt_length = len(inputs[0])
trajectory = []
for step, snapshot in enumerate(default_output.histories):
    token_ids = first_sequence(snapshot)
    generated_ids = token_ids[prompt_length:]
    masks = sum(token_id == tokenizer.mask_token_id for token_id in generated_ids)
    trajectory.append({
        "step": step,
        "mask_ratio": round(masks / max(len(generated_ids), 1), 3),
        "text": tokenizer.decode(generated_ids, skip_special_tokens=False),
    })

selected_steps = sorted({0, len(trajectory) // 2, len(trajectory) - 1})
for step in selected_steps:
    row = trajectory[step]
    print(f"t={row['step']:>3} · mask ratio={row['mask_ratio']:.3f}")
    print(row["text"], "\n")

## 03 · Training recipe

The workshop does not attempt full Tiny-A2D training. We audit an official warmup recipe and reduce it to a two-step smoke run.

In [ ]:
SOURCE_MODEL = "Qwen/Qwen3-0.6B"
CONVERTED_MODEL = ".models/a2d/Qwen3-0.6B"

conversion_command = (
    f"python dllm/pipelines/a2d/convert.py --model_name_or_path {SOURCE_MODEL} "
    f"--output_dir {CONVERTED_MODEL}"
)
training_smoke_command = (
    "WANDB_MODE=disabled accelerate launch "
    "--config_file scripts/accelerate_configs/ddp.yaml --num_processes 1 "
    "examples/a2d/mdlm/pt.py "
    f"--model_name_or_path {CONVERTED_MODEL} "
    "--dataset_args Trelis/tiny-shakespeare --text_field Text --insert_eos False "
    "--max_length 128 --max_steps 2 --learning_rate 1e-4 "
    "--per_device_train_batch_size 1 --per_device_eval_batch_size 1 "
    "--eval_strategy no --report_to none "
    "--output_dir .models/a2d/Qwen3-0.6B/kdd2026-smoke"
)

print("1. Convert the source model:\n", conversion_command)
print("\n2. Run the two-step smoke test:\n", training_smoke_command)

## 04 · Unified evaluation

Use the official Tiny-A2D GSM8K configuration for the reproducibility run. The full evaluation is launched from a terminal after the command is reviewed.

In [ ]:
evaluation_command = (
    "accelerate launch --num_processes 4 dllm/pipelines/a2d/eval.py "
    "--tasks gsm8k_cot --model a2d_mdlm --apply_chat_template --num_fewshot 0 "
    f"--model_args 'pretrained={MODEL_ID},max_new_tokens=256,steps=256,"
    "block_size=256,cfg_scale=0.0,temperature=0.0'"
)

print(evaluation_command)
reference_score = 29.3  # reproduced GSM8K score in examples/a2d/README.md
observed_score = None   # enter the workshop run after evaluation
absolute_delta = (
    abs(observed_score - reference_score)
    if observed_score is not None else None
)
{"reference": reference_score, "observed": observed_score, "delta": absolute_delta}

## 05 · Extend the scheduler

Implement the real dLLM scheduler interface, test its endpoints, then pass it to the same sampler.

In [ ]:
from dataclasses import dataclass
import torch
from dllm.core.schedulers.alpha import BaseAlphaScheduler

@dataclass
class QuadraticAlphaScheduler(BaseAlphaScheduler):
    def _alpha(self, t):
        return 1 - t**2

    def _alpha_derivative(self, t):
        return -2 * t

quadratic_scheduler = QuadraticAlphaScheduler()
probe_times = torch.linspace(0, 1, 5)
alpha_values = quadratic_scheduler.alpha(probe_times)

assert torch.isclose(alpha_values[0], torch.tensor(1.0))
assert torch.isclose(alpha_values[-1], torch.tensor(0.0))
assert torch.all(alpha_values[:-1] >= alpha_values[1:])

custom_sampler = dllm.core.samplers.MDLMSampler(
    model=model, tokenizer=tokenizer, scheduler=quadratic_scheduler
)
alpha_values

## 06 · Troubleshooting and audit

Record the environment and results only after the teaching workflow is complete.

In [ ]:
from pathlib import Path
from pprint import pprint
import json
import platform

audit_record = {
    "model_id": MODEL_ID,
    "seed": SEED,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "default_config": asdict(sampler_config),
    "default_output": default_text,
    "default_latency_s": round(default_latency, 3),
    "comparison_results": comparison_results,
    "trajectory": trajectory,
    "evaluation_reference": reference_score,
    "evaluation_observed": observed_score,
}

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
record_path = artifact_dir / "kdd2026-dllm-experiment-record.json"
record_path.write_text(json.dumps(audit_record, indent=2), encoding="utf-8")
pprint({"saved": str(record_path), "cuda_available": audit_record["cuda_available"]})

### Final audit checklist

- Pin the final KDD repository tag and Hugging Face model revision.
- Run the default inference and parameter comparison on the tutorial GPU.
- Verify the two-step training command from a clean checkout.
- Run the full evaluation once before publishing a reference tolerance.
- Restart the kernel and execute the notebook top to bottom.
- Clear private paths, credentials, and large outputs before release.

This remains one combined notebook because all six modules share the same model, tokenizer, sampler, and experiment state.